# 07 — LightGBM, the general model (Phase 4)

The first trained model. It sees **every** row and is asked one question: will
this station reach the tier within the horizon?

### What to expect, and why it is not a disappointment

This model will look excellent and will mostly be a monitor.

Roughly two thirds of positive rows at the 1-hour horizon are roads that are
*already* flooded. Predicting those correctly is easy — the water is right there
in `fl_depth_now`. A model optimising a single number will spend its capacity
there, because that is where the rows are, and it will post a high recall that
means very little operationally.

So the number to watch in this notebook is **not** PR-AUC or recall. It is:

- **`recall_onset`** — recall on rows where the road was still dry
- **`event_pod`** — fraction of real floods caught *before the water arrived*
- **`median_lead_minutes`** — how much warning, when it did catch one

Persistence — "it will be as it is now" — already scores 0.67 PR-AUC and 64%
recall. If this model lands near that, it has learned to be a thermometer.

Notebook 08 builds the specialist that is supposed to fix it.

### What you need

`05_features.ipynb` and `06_baselines.ipynb` must have been run.
LightGBM must be installed (`pip install -r requirements.txt`).
About 12 minutes.

## Setup

In [1]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_here = Path.cwd()
_root = next(p for p in [_here, *_here.parents] if (p / "config/config.yaml").is_file())
sys.path.insert(0, str(_root / "src"))
os.chdir(_root)

import numpy as np
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

from bkkflood.config import load_config
from bkkflood.rawio import connect
from bkkflood.evaluate import folds
from bkkflood.models import run_fold, gain_importance

CFG = load_config()
FOLDS = folds()
TIERS = sorted(int(t) for t in CFG["flood_event"]["tiers_cm"].values())
HORIZONS = CFG["horizons_hours"]
BASE = pd.read_parquet("data/features/baseline_results.parquet")
con = connect()
print("folds:", [f["test"] for f in FOLDS], "| tiers:", TIERS, "| horizons:", HORIZONS)

folds: [2022, 2023, 2024, 2025] | tiers: [5, 15, 30] | horizons: [1, 3, 6]


## 1. Train every fold, tier and horizon

36 models: 4 folds × 3 tiers × 3 horizons. Each takes roughly 12 seconds.

**What happens inside `run_fold`, in order:** train on the training years,
early-stop on the validation year, pick the alert threshold on the **validation**
year, then apply that threshold unchanged to the test year. That third step is
the one usually skipped — choosing a threshold after seeing the test set is
leakage of the *decision*, and it is quietly worth several points.

The 30 cm tier has around 43 positive rows in a year. It is reported for
completeness, but a model trained on 43 examples is not a model, and the results
there should be read as "we do not have the data" rather than as a score.

In [2]:
results = []
t0 = time.time()
for fold in FOLDS:
    for tier in TIERS:
        for h in HORIZONS:
            try:
                row, booster, meta = run_fold(fold, tier, h, "general", con=con)
                results.append(row)
            except Exception as e:
                print(f"  SKIP tier={tier} h={h} fold={fold['test']}: {type(e).__name__}: {e}")
    print(f"fold tested on {fold['test']} done  ({time.time()-t0:.0f}s)")

GEN = pd.concat(results, ignore_index=True)
GEN.to_parquet("data/features/model_general_results.parquet", index=False)
print("\n", GEN.shape, "-> data/features/model_general_results.parquet")

fold tested on 2022 done  (150s)
fold tested on 2023 done  (260s)
fold tested on 2024 done  (407s)
fold tested on 2025 done  (566s)

 (36, 29) -> data/features/model_general_results.parquet


## 2. Against the baselines

The headline setting: 15 cm, 1 hour ahead, averaged across folds.

In [3]:
def compare(tier, h):
    cols = ["pr_auc", "precision", "recall", "recall_onset", "f2", "fnr"]
    g = (GEN[(GEN.tier_cm == tier) & (GEN.horizon_h == h)][cols]
         .mean().to_frame("lightgbm_general").T)
    b = (BASE[(BASE.tier_cm == tier) & (BASE.horizon_h == h)]
         .groupby("baseline")[cols].mean())
    return pd.concat([g, b]).round(4)

print("15 cm, 1 hour ahead\n")
print(compare(15, 1).to_string())

15 cm, 1 hour ahead

                  pr_auc  precision  recall  recall_onset      f2     fnr
lightgbm_general  0.5656     0.6305  0.5581        0.1899  0.5708  0.4419
always_negative   0.0004     0.0000  0.0000        0.0000  0.0000  1.0000
climatology       0.0009     0.0019  0.1037        0.0952  0.0085  0.8963
persistence       0.5529     0.6175  0.5443        0.1635  0.5541  0.4557
rain_rule         0.1035     0.1066  0.3154        0.2180  0.2185  0.6846


In [4]:
pers = BASE[(BASE.tier_cm == 15) & (BASE.horizon_h == 1) &
            (BASE.baseline == "persistence")]
g = GEN[(GEN.tier_cm == 15) & (GEN.horizon_h == 1)]

print("The comparison that matters — 15 cm / 1 h\n")
for label, a, b in [
    ("PR-AUC          ", g.pr_auc.mean(), pers.pr_auc.mean()),
    ("overall recall  ", g.recall.mean(), pers.recall.mean()),
    ("ONSET recall    ", g.recall_onset.mean(), pers.recall_onset.mean()),
]:
    delta = a - b
    verdict = "better" if delta > 0.005 else ("about the same" if abs(delta) <= 0.005 else "WORSE")
    print(f"  {label}  model {a:.4f}   persistence {b:.4f}   ({verdict})")

print()
print("If overall recall improves but onset recall does not, the model has only")
print("gotten better at describing water that is already on the road.")

The comparison that matters — 15 cm / 1 h

  PR-AUC            model 0.5656   persistence 0.5529   (better)
  overall recall    model 0.5581   persistence 0.5443   (better)
  ONSET recall      model 0.1899   persistence 0.1635   (better)

If overall recall improves but onset recall does not, the model has only
gotten better at describing water that is already on the road.


## 3. What the model actually used

Read this table against the point above. If the top of it is `fl_depth_now` and
its lags and `fl_hours_since_*`, the model is reading the sensor rather than
forecasting from rainfall and terrain.

That is not cheating and it is not a bug — those really are the most predictive
columns for the question as posed. It is the reason the question has to be
re-posed, which is notebook 08.

In [5]:
row, booster, meta = run_fold(FOLDS[-1], 15, 1, "general", con=con,
                              save_as="general_t15_h1_final")
imp = gain_importance(booster, 15)
print(f"most recent fold (test {FOLDS[-1]['test']}), 15 cm / 1 h\n")
print(imp.round(4).to_string(index=False))

autoreg = imp[imp.feature.str.startswith("fl_")].gain_share.sum()
print(f"\nshare of total gain from the station's own depth history: {autoreg:.1%}")
print("rain, terrain, canal and calendar features share the rest.")

most recent fold (test 2025), 15 cm / 1 h

            feature        gain  splits  gain_share
       fl_depth_now 422547.0895      65      0.6583
     rain_rf1hr_max  56158.0715      34      0.0875
    rain_rf1hr_mean  22748.7667      30      0.0354
fl_hours_since_15cm  16368.1066     172      0.0255
 rain_rf1hr_delta1h  14426.7588      45      0.0225
          fl_std_3h  11355.4477      30      0.0177
        cal_doy_cos   7056.5675     214      0.0110
        cal_doy_sin   6663.7399     224      0.0104
   tide_spring_neap   6262.7085     205      0.0098
         fl_max_24h   6019.2537      66      0.0094
 fl_hours_since_5cm   5629.0777      96      0.0088
       cal_hour_sin   4759.8086     153      0.0074
water_offline_share   4065.0354     123      0.0063
        tide_m2_cos   3930.6892     116      0.0061
 water_rising_share   3923.2840     127      0.0061

share of total gain from the station's own depth history: 72.0%
rain, terrain, canal and calendar features share the rest.


## 4. Does it get harder further out?

It must. Scores that hold up across horizons mean a feature knows more than it
should — the check exists because it is the cheapest possible leakage detector.

In [6]:
decay = GEN.groupby(["tier_cm", "horizon_h"])[
    ["pr_auc", "recall", "recall_onset", "event_pod", "median_lead_minutes"]
].mean().round(4)
print(decay.to_string())

t15 = decay.loc[15]
print()
if t15.pr_auc.is_monotonic_decreasing and t15.recall_onset.is_monotonic_decreasing:
    print("Both fall with horizon, as they must.")
else:
    print("WARNING: something does not decay with horizon. Investigate before")
    print("reporting any of these numbers.")

                   pr_auc  recall  recall_onset  event_pod  median_lead_minutes
tier_cm horizon_h                                                              
5       1          0.5192  0.4911        0.0634     0.1384                15.00
        3          0.2712  0.2627        0.0422     0.1917                15.00
        6          0.1693  0.2276        0.1086     0.3582                37.50
15      1          0.5656  0.5581        0.1899     0.4870                15.00
        3          0.2739  0.2963        0.0853     0.5710                15.00
        6          0.1542  0.1722        0.0442     0.5487                15.00
30      1          0.5264  0.5807        0.2744     0.6421                15.00
        3          0.2644  0.3260        0.1353     0.7972                18.75
        6          0.1662  0.1987        0.0797     0.8353                22.50

Both fall with horizon, as they must.


## 5. Events and warning time

The number BMA would actually care about: of the real floods, how many were
flagged **before the water arrived**, and how much notice came with them.

An alert at the moment the road floods is detection. It is not warning.

In [7]:
ev = GEN[GEN.tier_cm == 15].groupby("horizon_h")[
    ["events", "event_pod", "median_lead_minutes"]].mean().round(3)
print("15 cm tier, mean across folds\n")
print(ev.to_string())
print()
print("Median lead is reported in minutes. The modelling cadence is 15 minutes,")
print("so a median of 15 means the typical warning was a single time step —")
print("real, but not much of a head start.")

15 cm tier, mean across folds

           events  event_pod  median_lead_minutes
horizon_h                                        
1           129.0      0.487                 15.0
3           129.0      0.571                 15.0
6           129.0      0.549                 15.0

Median lead is reported in minutes. The modelling cadence is 15 minutes,
so a median of 15 means the typical warning was a single time step —
real, but not much of a head start.


In [8]:
out = Path("docs/reports/model_general.md")
out.parent.mkdir(parents=True, exist_ok=True)
with out.open("w") as f:
    f.write("# LightGBM — general model\n\n")
    f.write("Generated by `notebooks/07_train_lightgbm.ipynb`. Thresholds chosen on\n")
    f.write("the validation year and applied unchanged to the test year.\n\n")
    f.write("## Against the baselines: 15 cm, 1 hour\n\n")
    f.write(compare(15, 1).to_markdown())
    f.write("\n\n## All tiers and horizons\n\n")
    f.write(GEN.groupby(["tier_cm", "horizon_h"])[
        ["pr_auc", "base_rate", "precision", "recall", "recall_onset",
         "f2", "fnr", "event_pod", "median_lead_minutes"]].mean().round(4).to_markdown())
    f.write("\n\n## Per fold, 15 cm / 1 h\n\n")
    f.write(GEN[(GEN.tier_cm == 15) & (GEN.horizon_h == 1)][
        ["test_year", "train_years", "pr_auc", "precision", "recall",
         "recall_onset", "event_pod"]].round(4).to_markdown(index=False))
    f.write("\n")
print("wrote", out)

wrote docs/reports/model_general.md


## What this notebook establishes

A general model trained on every row. Whatever its headline numbers, its job in
this project is to be **the thing the onset specialist is compared against** —
and to make concrete why a specialist is needed at all.

**Next:** `08_train_onset.ipynb` asks the harder question, on the rows where the
road is still dry.